## Torch dependencies

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

C:\Users\Daniel\Documents\Personal\breast_cancer_detection\.venv\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## MLFlow dependencies and initialization

In [3]:
import mlflow
from pprint import pprint

In [4]:
# Set here the URI from your MLFLow Tracking Server
TRACKING_URI = "http://localhost:5000"
client = mlflow.MlflowClient(tracking_uri=TRACKING_URI)
mlflow.set_tracking_uri(TRACKING_URI)

### Experiment creation

In [5]:
experiment_description = (
    "Training ResNet18 CNN for breat cancer detection."
    "This approach uses MLFlow instead of the custom pipeline built before."
    "No_HT stands for No Hyperparameter Tunning"
)

experiment_tags={
    "project_name": "breat-cancer-dection",
    "model_name": "resnet50",
    "mlflow.note.content": experiment_description
}

experiment_name = "ResNet_BreastCancer_No_HT"

exp = client.get_experiment_by_name(experiment_name)

if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name, tags=experiment_tags
    )
else:
    exp_id = exp.experiment_id
    print(f"Experiment {experiment_name} already exists. Skipping creation")

Experiment ResNet_BreastCancer_No_HT already exists. Skipping creation


## Training Dependencies

In [6]:
from pathlib import Path

# Force add the project root to sys.path (adjust as needed)
project_root = Path("../").resolve()  # one level up from /notebooks/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from tqdm import tqdm

from datasets.cbisddsm import CBISDDSMDataset
from training.early_stopping import EarlyStopping
from training.engine import train_epoch, evaluate_epoch
from training.focal_loss import FocalLoss
from utils.to_tensor_16b import ToFloatTensor16Bit

import pyarrow.parquet as pq
import pandas as pd
import numpy as np

import os
import time
import uuid

In [7]:
# Path to the dataset file
DATA_ABS_PATH     = os.path.abspath("D:/tfm/data")
IMAGES_ABS_PATH   = os.path.abspath("D:/tfm/data/CBIS-DDSM")
PNG_ABS_PATH      = os.path.abspath("D:/tfm/data/CBIS-DDSM-PNG")
CBISDDSM_FIXED_SET = DATA_ABS_PATH + '/meta/CBIS-DDSM-fixed.parquet'

### Set hyperparameters

In [8]:
num_epochs          = 50
train_batch_size    = 64
test_batch_size     = train_batch_size * 2
val_batch_size      = train_batch_size * 2
prefetch_factor     = 2
num_workers         = 8

learning_rate_l4    = 6.408966256267519e-05
learning_rate_fc    = 6.005493192543597e-05
scheduler_patience  = 10
alpha               = [1.0, 2.8619563143873994, 1.0]
early_stop_patience = 16
gamma               = 1.1120913750166153
dropout_rate        = 0.696670579337237
weight_decay        = 8.683454550720445e-05

early_stop_metric   = "val_recall"
early_stop_delta    = 0.001
early_stop_mode     = "max"

resize              = 224
horizontal_flip     = 0.5
degrees             = 10
# brightness          = 0.2
# contrast            = 0.2
kernel_size         = 3
normalize_mean      = [0.5, 0.5, 0.5]
normalize_std       = [0.5, 0.5, 0.5]

multi_view          = False

In [9]:
transform = transforms.Compose([
    transforms.Resize((resize, resize)),
    transforms.RandomHorizontalFlip(horizontal_flip),
    transforms.RandomRotation(degrees=degrees),
    # transforms.ColorJitter(brightness=brightness, contrast=contrast),
    ToFloatTensor16Bit(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std)
])

eval_transform = transforms.Compose([
    transforms.Resize((resize, resize)),
    ToFloatTensor16Bit(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std)
])

### Set DataLoaders

In [10]:
# Dataframes
train_df = pd.read_parquet("train_png.parquet")
val_df = pd.read_parquet("val_png.parquet")
test_df = pd.read_parquet("test_png.parquet")

# PyTorch datasets
train_dataset = CBISDDSMDataset("train_png.parquet", transform=transform, multi_view=multi_view, images_base_path=PNG_ABS_PATH)
val_dataset   = CBISDDSMDataset("val_png.parquet", transform=eval_transform, multi_view=multi_view, images_base_path=PNG_ABS_PATH)
test_dataset  = CBISDDSMDataset("test_png.parquet", transform=eval_transform, multi_view=multi_view, images_base_path=PNG_ABS_PATH)

# PyTorch dataloaders
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset, batch_size=val_batch_size, shuffle=False, num_workers=num_workers, pin_memory=True,  persistent_workers=True)
test_loader  = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, num_workers=num_workers, pin_memory=True,  persistent_workers=True)

# MLFlow datasets
ml_train_dataset = mlflow.data.from_pandas(train_df, name="cbis_ddsm_train")
ml_val_dataset = mlflow.data.from_pandas(val_df, name="cbis_ddsm_val")
ml_test_dataset = mlflow.data.from_pandas(test_df, name="cbis_ddsm_test")

## Using my stuff for training

In [11]:
print(f"\n==== Training ====")

correlation_id = str(uuid.uuid4())
REGISTERED_MODEL_NAME = "resnet50-breast-cancer"

with mlflow.start_run(experiment_id=exp_id, log_system_metrics=True) as run:
    mlflow.log_params(
            params={
                "correlation_id":      correlation_id,
                "num_epochs":          num_epochs,
                "test_batch_size":     test_batch_size,
                "train_batch_size":    train_batch_size,
                "val_batch_size":      val_batch_size,
                "prefetch_factor":     prefetch_factor,
                "num_workers":         num_workers,
                "learning_rate_l4":    learning_rate_l4,
                "learning_rate_fc":    learning_rate_fc,
                "scheduler_patience":  scheduler_patience,
                "alpha":               alpha,
                "early_stop_patience": early_stop_patience,
                "gamma":               gamma,
                "dropout_rate":        dropout_rate,
                "early_stop_metric":   early_stop_metric,
                "early_stop_delta":    early_stop_delta,
                "early_stop_mode":     early_stop_mode,
                "resize":              resize,
                "horizontal_flip":     horizontal_flip,
                "degrees":             degrees,
                "kernel_size":         kernel_size,
                "normalize_mean":      normalize_mean,
                "normalize_std":       normalize_std,
                "weight_decay":        weight_decay,
                "multi_view":          multi_view
            }
        )

    # Model initialization
    model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)

    # Freezing model layers
    for param in model.parameters():
        param.requires_grad = False

    model.fc = torch.nn.Sequential(
        torch.nn.Dropout(dropout_rate),
        torch.nn.Linear(model.fc.in_features, 3)
    )
    
    for layer in [model.layer4, model.fc]:
        for param in layer.parameters():
            param.requires_grad = True

    model = model.to(device)
    
    optimizer = Adam([
        {'params': model.layer4.parameters(), 'lr': learning_rate_l4},
        {'params': model.fc.parameters(), 'lr': learning_rate_fc}
    ], weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=scheduler_patience, min_lr=1e-6)
    criterion = FocalLoss(gamma=gamma, alpha=alpha)
    early_stopping = EarlyStopping(
        monitor=early_stop_metric,
        mode=early_stop_mode,
        patience=early_stop_patience,
        delta=early_stop_delta
    )

    # Metric to prioritize
    best_malignant_recall = 0
    best_model_wts = None
    
    for epoch in range(num_epochs):
        start_time = time.time()
        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        # Training for current epoch
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate model for current epoch
        val_acc, val_loss, val_recall, val_precision, val_f1, val_auc, val_views, val_view_predictions, val_class_metrics  = evaluate_epoch(
            model, val_loader, criterion, device
        )

        val_malignant_recall = val_class_metrics.get('MALIGNANT', {}).get('recall', 0) or \
                              val_class_metrics.get(2, {}).get('recall', 0) or \
                              val_class_metrics.get('class_2', {}).get('recall', 0)

        if val_malignant_recall > best_malignant_recall:
            best_malignant_recall = val_malignant_recall
            best_model_wts = model.state_dict()

        # Saving epoch metrics to MLFlow
        mlflow.log_metric(key="train_loss", value=train_loss, step=epoch)
        mlflow.log_metric(key="val_loss", value=val_loss, step=epoch)
        mlflow.log_metric(key="val_accuracy", value=val_acc, step=epoch)
        mlflow.log_metric(key="val_recall", value=val_recall, step=epoch)
        mlflow.log_metric(key="val_precision", value=val_precision, step=epoch)
        mlflow.log_metric(key="val_f1", value=val_f1, step=epoch)
        mlflow.log_metric(key="val_auc", value=val_auc, step=epoch)
        mlflow.log_metric("val_malignant_recall", val_malignant_recall, step=epoch)

        # Log per-class metrics
        for class_name, metrics in val_class_metrics.items():
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(f"val_{class_name}_{metric_name}", metric_value, step=epoch)

        # Learning Rate Scheduler
        scheduler.step(val_malignant_recall)

        # Checking early stop
        if early_stopping.step(val_recall):
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    # Evaluating the best trained model
    model.load_state_dict(best_model_wts)

    # Test evaluation
    test_acc, test_loss, test_recall, test_precision, test_f1, test_auc, test_views, test_view_predictions, test_class_metrics  = evaluate_epoch(
        model, test_loader, criterion, device
    )
    
    # Saving test metrics to MLFlow
    mlflow.log_metric(key="test_loss", value=test_loss)
    mlflow.log_metric(key="test_accuracy", value=test_acc)
    mlflow.log_metric(key="test_recall", value=test_recall)
    mlflow.log_metric(key="test_precision", value=test_precision)
    mlflow.log_metric(key="test_f1", value=test_f1)
    mlflow.log_metric(key="test_auc", value=test_auc)

    for class_name, metrics in val_class_metrics.items():
                    for metric_name, metric_value in metrics.items():
                        mlflow.log_metric(f"test_{class_name}_{metric_name}", metric_value)
    
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="pytorch_model"
    )

    from inference.resnet_image_predictor import ResNetImagePredictor
    import tempfile
    import os
    
    with tempfile.TemporaryDirectory() as tmpdir:
        temp_model_path = os.path.join(tmpdir, "temp_pytorch_model")
        mlflow.pytorch.save_model(model, temp_model_path)
        
        mlflow.pyfunc.log_model(
            artifact_path="image_predictor",
            python_model=ResNetImagePredictor(),
            artifacts={"pytorch_model": temp_model_path},
            pip_requirements=[
                f'mlflow=={mlflow.__version__}',
                f'torch=={torch.__version__}',
                f'torchvision=={torchvision.__version__}',
                'pillow', 'pydicom', 'numpy', 'pandas', 'opencv-python'
            ]
        )
    
    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/image_predictor"
    mlflow.register_model(
        model_uri=model_uri,
        name=REGISTERED_MODEL_NAME
    )
    
    print(f"\n{'='*60}")
    print(f"Model registered: {REGISTERED_MODEL_NAME}")
    print(f"Run ID: {run_id}")
    print(f"Correlation ID: {correlation_id[:6]}")
    print(f"\nTest Results:")
    print(f"   Accuracy:  {test_acc:.4f}")
    print(f"   Recall:    {test_recall:.4f}")
    print(f"   Precision: {test_precision:.4f}")
    print(f"   F1:        {test_f1:.4f}")
    print(f"   AUC:       {test_auc:.4f}")
    print(f"\nServe with:")
    print(f'$env:MLFLOW_TRACKING_URI="http://localhost:5000"; mlflow models serve -m "models:/{REGISTERED_MODEL_NAME}/latest" --port 5003 --env-manager=local')
    print(f"{'='*60}")

mlflow.end_run()

2025/11/16 20:23:59 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.



==== Training ====
  Error =>  [WinError 2] The system cannot find the file specified

Epoch 1/50


Training:   0%|          | 0/37 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]


Epoch 2/50


...

Early stopping triggered at epoch 23


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

2025/11/16 20:46:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/16 20:46:32 WARNING mlflow.utils.requirements_utils: Found torch version (2.8.0+cu129) contains a local version label (+cu129). MLflow logged a pip requirement for this package as 'torch==2.8.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/11/16 20:46:36 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.23.0+cu129) contains a local version label (+cu129). MLflow logged a pip requirement for this package as 'torchvision==0.23.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/11/16 20:46:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example`

2025/11/16 20:46:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'resnet50-breast-cancer' already exists. Creating a new version of this model...
2025/11/16 20:46:49 WARNING mlflow.tracking._model_registry.fluent: Run with id 4d1e5b68513a4cc993ebf5fcaf096b7f has no artifacts at artifact path 'image_predictor', registering model based on models:/m-4835032633ba4781957a89c0674db18a instead
2025/11/16 20:46:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: resnet50-breast-cancer, version 4
Created version '4' of model 'resnet50-breast-cancer'.
2025/11/16 20:46:49 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2025/11/16 20:46:49 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitorin


Model registered: resnet50-breast-cancer
Run ID: 4d1e5b68513a4cc993ebf5fcaf096b7f
Correlation ID: 8b6390

Test Results:
   Accuracy:  0.5476
   Recall:    0.5532
   Precision: 0.5585
   F1:        0.5558
   AUC:       0.7443

Serve with:
$env:MLFLOW_TRACKING_URI="http://localhost:5000"; mlflow models serve -m "models:/resnet50-breast-cancer/latest" --port 5003 --env-manager=local
🏃 View run fortunate-lark-954 at: http://localhost:5000/#/experiments/26/runs/4d1e5b68513a4cc993ebf5fcaf096b7f
🧪 View experiment at: http://localhost:5000/#/experiments/26
